# Week 3, day 3 (afternoon) — Worksheet 04 SOLUTIONS: what COPY INTO actually does

Executed in the lab image against the real lab CSVs; every quoted figure was
observed.

The through-line: each `FILE_FORMAT` setting exists because some specific thing
in the file would otherwise be parsed wrong — and a wrong parse mostly does not
announce itself.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 04 — What COPY INTO does. Run this once.
import pandas as pd

PRODUCTS = "data/products_2013_01_01.csv"
SALES = "data/sales_2013_01_01.csv"

# The 14 positions the COPY selects, in order, named as STG_Sales declares them.
STG_SALES_COLS = [
    "TRANS_ID", "PROD_KEY", "STORE_KEY", "TRANS_DT", "TRANS_TIME",
    "PRIORITY", "SALES_QTY", "SALES_PRICE", "SALES_AMT",
    "DISCOUNT", "SALES_COST", "SALES_MGRN", "SHIP_MODE", "SHIP_COST",
]

print("rebuilding the COPY INTO in", len(STG_SALES_COLS), "positions + BATCH_ID")

PART A — SKIP_HEADER and positional selection

### Question 1

`SKIP_HEADER = 1` plus `SELECT $1 ... $14` means: throw the header away, then take fields by position. Reproduce that with `header=None`, `skiprows=1` and `names=STG_SALES_COLS`. Print the shape and the first two rows of `TRANS_ID`, `TRANS_DT` and `SHIP_MODE`.
> **NOTE:** the CSV header says `SHIPMODE`; you are naming that position `SHIP_MODE`, exactly as the COPY does. Positional loading lets you rename on arrival.

In [ ]:
stg = pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)
print("shape:", stg.shape)
print()
print(stg[["TRANS_ID", "TRANS_DT", "SHIP_MODE"]].head(2).to_string(index=False))

```
shape: (100000, 14)

 TRANS_ID   TRANS_DT     SHIP_MODE
   247136   3/7/2010 "Regular Air"
   247138 12/22/2009 "Regular Air"
```

100,000 rows, 14 columns, and the column is called `SHIP_MODE` even though the
file's header says `SHIPMODE` — because nothing read that header. `skiprows=1`
discarded it and `names=` supplied the names, which is precisely what
`SKIP_HEADER = 1` plus `SELECT $1 ... $14` does.

Note also what has *not* happened yet: `TRANS_DT` is still the string
`3/7/2010`, and `SHIP_MODE` still carries its quotes. Positional selection moves
fields into columns. Everything else is the other three settings' job.

### Question 2

Confirm nothing was lost. Count the data lines in the raw file with `open()` (all lines minus the header) and compare with `len(stg)`.

In [ ]:
with open(SALES) as fh:
    raw_lines = sum(1 for _ in fh)
stg = pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)
print("lines in file:      ", raw_lines)
print("minus the header:   ", raw_lines - 1)
print("rows loaded:        ", len(stg))
print("match:", raw_lines - 1 == len(stg))

```
lines in file:       100001
minus the header:    100000
rows loaded:         100000
match: True
```

Exactly the arithmetic `SKIP_HEADER = 1` promises. Worth doing once per new
source: it is the cheapest possible check, and it catches the two failures that
are otherwise invisible — an embedded newline splitting one record into two, and
a trailing blank line adding a row of nulls.

It is also the *only* thing `4_Validate_stage_tables.sql` checks. Keep that in
mind through Q10.

### Question 3

Now do it wrong: load the same file with `header=None` but **without** `skiprows`. Print the first row and the dtype of `TRANS_ID`.
> **NOTE:** this is a COPY with `SKIP_HEADER` left off. Snowflake would reject the row rather than accept it as text, but the lesson is the same — one setting decides whether the header is data.

In [ ]:
bad = pd.read_csv(SALES, header=None, names=STG_SALES_COLS)
print("row 0:", bad.iloc[0].tolist()[:5])
print()
print("TRANS_ID dtype:", bad["TRANS_ID"].dtype)
print("rows:", len(bad), "vs", len(pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)))

Without `skiprows`, the header becomes a row of data:

```
row 0: ['TRANS_ID', 'PROD_KEY', 'STORE_KEY', 'TRANS_DT', 'TRANS_TIME']

TRANS_ID dtype: object
rows: 100001 vs 100000
```

One extra row — and the damage is not the row. It is `dtype: object`. A single
non-numeric value anywhere in a column forces the whole column to Python objects,
so all 100,001 values in `TRANS_ID` are now text, including the 100,000 that are
perfectly good integers.

Snowflake would behave differently here: `STG_Sales.TRANS_ID` is declared
`INTEGER`, so the string `'TRANS_ID'` fails conversion and the COPY rejects that
row (or aborts, depending on `ON_ERROR`). The typed table catches it. pandas has
no declared schema to check against, so it widens the type and carries on — which
is why the next question is possible at all.

### Question 4

That bad load emitted a `DtypeWarning` about mixed types. Count the actual Python types in `bad["SALES_AMT"]` with `.map(type).value_counts()`, print the last position holding a `str` and the first holding a `float`, then total the column. **The total is supposed to fail.**
> **NOTE:** the split is not where the bad row is. Look at the number and ask what else in pandas has that size.

In [ ]:
bad = pd.read_csv(SALES, header=None, names=STG_SALES_COLS)
col = bad["SALES_AMT"]
print("declared dtype:", col.dtype)
print(col.map(type).value_counts().to_string())
print()
is_str = col.map(lambda v: isinstance(v, str))
print("last str position: ", int(is_str.to_numpy().nonzero()[0].max()))
print("first float position:", int((~is_str).to_numpy().nonzero()[0].min()))
print()
print(col.sum())

The types split at a suspiciously round number:

```
declared dtype: object
SALES_AMT
<class 'str'>      65536
<class 'float'>    34465

last str position:  65535
first float position: 65536
```

**65,536 is 2^16** — it is pandas' read chunk size, not anything in the file.

`read_csv` infers dtypes per chunk (`low_memory=True` is the default). Chunk 1
contains row 0, the header text, so every value in that chunk is kept as `str`.
Chunks 2 and 3 contain no text, so their values are parsed as `float`. The
result is one column holding 65,536 strings followed by 34,465 floats — a type
boundary at an offset that has nothing to do with the data, and would move if
the file were a different size.

Then the total:

```
TypeError: can only concatenate str (not "float") to str
```

`sum()` started concatenating strings — `'SALES_AMT' + '49.61' + ...` is valid
Python — and crashed on reaching position 65,536, where the values became
floats. Had the file been under 65,536 rows, `sum()` would have **succeeded** and
returned a 400 KB string of digits.

That is the shape of the danger. The error is not the problem; the error is the
lucky outcome. The problem is a column whose type depends on how many rows
preceded it.

pandas does warn — `DtypeWarning: Columns (...) have mixed types` — and the fix
it suggests, `low_memory=False`, makes inference consistent across the file. But
the real fix is upstream: do not let the header become data.

PART B — FIELD_OPTIONALLY_ENCLOSED_BY, and why products does not use it

### Question 5

The sales FILE_FORMAT sets `FIELD_OPTIONALLY_ENCLOSED_BY = '"'`; the products FILE_FORMAT does not. Print the first raw line of data from each file and explain the difference in one sentence.

In [ ]:
for path in (PRODUCTS, SALES):
    with open(path) as fh:
        fh.readline()             # header
        print("%-28s %s" % (path.split("/")[-1], fh.readline().rstrip()[:104]))
print()
print("products quote characters in row 1:", open(PRODUCTS).readlines()[1].count('"'))
print("sales    quote characters in row 1:", open(SALES).readlines()[1].count('"'))

```
products_2013_01_01.csv      657768,Product-657768,1.22,28.60,brand-14,1,active,4,category-4,1,subcategory-1
sales_2013_01_01.csv         247136,1021964,8106,3/7/2010,19,"""Medium""",0.9,30.98,49.61,0,72.8,-41.82,"""Regular Air""",17.08

products quote characters in row 1: 0
sales    quote characters in row 1: 12
```

The products file quotes nothing; the sales file wraps its two text columns in
three quotes a side. So the two FILE_FORMATs differ for a reason — products has
no enclosure to declare, and declaring one would change nothing.

The asymmetry is worth noticing rather than copying. A FILE_FORMAT is a claim
about a specific file's layout. These two files came from the same extract on
the same day and still needed different claims; the settings belong to the file,
not to the project.

And `FIELD_OPTIONALLY_ENCLOSED_BY = '"'` is correctly set for sales — it removes
the outer pair. Three quotes minus one pair leaves one pair, which is Q6.

### Question 6

Strip the surviving quotes the way a transformational SELECT would — `TRIM($6, '\"')`. Apply `.str.strip('\"')` to `PRIORITY` and `SHIP_MODE`, then print the distinct values of each with `repr()` and the row count for `PRIORITY == "High"`.

In [ ]:
stg = pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)
for col in ("PRIORITY", "SHIP_MODE"):
    stg[col] = stg[col].str.strip('"')
print("PRIORITY:")
for v in sorted(stg["PRIORITY"].unique()):
    print("   ", repr(v))
print("SHIP_MODE:")
for v in sorted(stg["SHIP_MODE"].unique()):
    print("   ", repr(v))
print()
print("rows where PRIORITY == 'High':", int((stg["PRIORITY"] == "High").sum()))

```
PRIORITY:
    'Critical'
    'High'
    'Low'
    'Medium'
    'Not Specified'
SHIP_MODE:
    'Delivery Truck'
    'Express Air'
    'Regular Air'

rows where PRIORITY == 'High': 21117
```

Clean values, and `WHERE PRIORITY = 'High'` now finds all 21,117 rows instead of
the zero it found in Worksheet 03.

Two things to take from this. First, where the fix goes: in the transformational
SELECT, as `TRIM($6, '"')`, so it happens once at load time. The alternative is
`WHERE PRIORITY = '"High"'` in every query forever, and eventually somebody
writes the natural version and gets an empty result.

Second, how you would have known. Five distinct values in one column, eight
across two — a `SELECT DISTINCT` on every low-cardinality text column is a few
seconds of work on arrival, and it is the check that would have caught this
before any query was written against the table.

PART C — DATE_FORMAT

### Question 7

`TRANS_DT` is declared `DATE` in `STG_Sales`, and the FILE_FORMAT sets `DATE_FORMAT = 'MM/DD/YYYY'`. Parse it with `format="%m/%d/%Y"`, print the dtype and min/max. Then parse the same column with `format="%d/%m/%Y"` inside a try/except and print what happens.
> **NOTE:** `3/7/2010` is a valid date under both readings. That is exactly the problem.

In [ ]:
stg = pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)
us = pd.to_datetime(stg["TRANS_DT"], format="%m/%d/%Y")
print("MM/DD/YYYY ->", us.dtype, "|", us.min().date(), "to", us.max().date())
try:
    eu = pd.to_datetime(stg["TRANS_DT"], format="%d/%m/%Y")
    print("DD/MM/YYYY ->", eu.dtype, "|", eu.min().date(), "to", eu.max().date())
except ValueError as exc:
    print("DD/MM/YYYY -> ValueError:", str(exc).split(". You might")[0])
print()
print("row 0 raw:", stg["TRANS_DT"].iloc[0], "-> US reading:", us.iloc[0].date())

```
MM/DD/YYYY -> datetime64[us] | 2009-01-01 to 2012-12-30
DD/MM/YYYY -> ValueError: time data "12/22/2009" doesn't match format "%d/%m/%Y"

row 0 raw: 3/7/2010 -> US reading: 2010-03-07
```

The declared format parses cleanly. The other one raises — but look at *what*
raised it: `12/22/2009`, a row with 22 in the second field. There is no
twenty-second month, so that row cannot be read as DD/MM.

One row's impossibility rejected the whole column. That is luck, not validation.
Row 0 is `3/7/2010`, which is 7 March under one reading and 3 July under the
other, and both are real dates. Nothing about that row would have complained.

This is why `DATE_FORMAT` is a required setting and not a nicety. A `DATE` column
loaded under the wrong convention contains valid dates, passes every null check,
sorts fine, and is wrong — with the error concentrated in the rows where day and
month happen to differ. Q8 counts them.

### Question 8

Find how many rows are ambiguous — where day and month are both 12 or less, so the string parses to a valid but *different* date under the other convention. Print the count and the percentage.

In [ ]:
stg = pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)
parts = stg["TRANS_DT"].str.split("/", expand=True).astype(int)
first, second = parts[0], parts[1]
ambiguous = (first <= 12) & (second <= 12) & (first != second)
n = int(ambiguous.sum())
print("rows where both fields are <= 12 and differ:", n, "(%.1f%%)" % (100 * n / len(stg)))
print("rows where the reading cannot change:      ", len(stg) - n)
print()
print("examples:", stg.loc[ambiguous, "TRANS_DT"].head(4).tolist())

```
rows where both fields are <= 12 and differ: 38142 (38.1%)
rows where the reading cannot change:       61858

examples: ['3/7/2010', '3/8/2011', '11/6/2010', '6/11/2011']
```

**38,142 rows — 38.1% — would silently land on a different date** under the other
convention. The remaining 61,858 have a day above 12 somewhere, so they are
self-disambiguating.

Read the examples: `11/6/2010` and `6/11/2011` are the same two numbers in
opposite order, and both are in this file. There is no rule you can infer from
the data to tell them apart. The only source of truth is the FILE_FORMAT setting
— which is to say, someone's knowledge of where the file came from.

Now put Q7 and Q8 together. Snowflake would have raised on the first
`12/22/2009` it hit and aborted the load, so this particular mistake would have
been caught. But that outcome depended entirely on the file happening to contain
a day above 12. A file covering only the first twelve days of a month contains no
such row — it loads clean, validates clean, and 38% of its dates are wrong.

Do not rely on the data to catch a configuration error. Confirm the format from
the source, and after loading, check that the date range matches what you expect
the file to cover.

PART D — BATCH_ID

### Question 9

The COPY appends a 15th column: `SPLIT_PART(METADATA$FILENAME, '/', -1)`. Given the staged path `data_loading_lab/csv_files/sales_2013_01_01.csv`, reproduce that in Python and add the result as `BATCH_ID`. Print the column list and the distinct `BATCH_ID` values.

In [ ]:
staged_path = "data_loading_lab/csv_files/sales_2013_01_01.csv"
batch_id = staged_path.split("/")[-1]          # SPLIT_PART(..., '/', -1)
stg = pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)
stg["BATCH_ID"] = batch_id
print("BATCH_ID =", repr(batch_id))
print("columns:", len(stg.columns))
print(list(stg.columns))
print()
print("distinct BATCH_ID:", stg["BATCH_ID"].unique().tolist())

```
BATCH_ID = 'sales_2013_01_01.csv'
columns: 15
['TRANS_ID', ..., 'SHIP_COST', 'BATCH_ID']

distinct BATCH_ID: ['sales_2013_01_01.csv']
```

`SPLIT_PART(METADATA$FILENAME, '/', -1)` takes the last `/`-separated piece of
the staged path — `data_loading_lab/csv_files/sales_2013_01_01.csv` becomes
`sales_2013_01_01.csv`. In Python, `.split("/")[-1]`.

`METADATA$FILENAME` is a pseudo-column Snowflake exposes on staged files, which
is the point of the transformational form: a plain `COPY INTO t FROM @stage`
cannot add a column that is not in the file, and this one is not in the file.

What it buys you is real. Load ten files into one table and every row knows which
file it came from. That is what makes a bad load reversible —
`DELETE FROM STG_Sales WHERE BATCH_ID = 'sales_2013_01_01.csv'` removes exactly
this load and nothing else. Without it, undoing a partial load means restoring
from a snapshot.

`INSERTED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP()` in the table definition adds
the other half — *when* — without the COPY mentioning it at all.

One distinct value here because one file was loaded. That count is a useful
thing to watch: it should equal the number of files you meant to load.

### Question 10

Build the finished staging table: positional load, quotes stripped, `TRANS_DT` parsed as a date, `BATCH_ID` added. Then print the row count grouped by `BATCH_ID` — the same audit `4_Validate_stage_tables.sql` runs — and, beside it, the row count grouped by the *year of* `TRANS_DT`.

In [ ]:
stg = pd.read_csv(SALES, header=None, skiprows=1, names=STG_SALES_COLS)
for col in ("PRIORITY", "SHIP_MODE"):
    stg[col] = stg[col].str.strip('"')
stg["TRANS_DT"] = pd.to_datetime(stg["TRANS_DT"], format="%m/%d/%Y")
stg["BATCH_ID"] = "sales_2013_01_01.csv"

print("what 4_Validate_stage_tables.sql reports:")
print(stg.groupby("BATCH_ID").size().rename("row_count").to_string())
print()
print("what the rows actually are:")
print(stg.groupby(stg["TRANS_DT"].dt.year).size().rename("row_count").to_string())

The table is now built the way the lab builds it. Here is the audit the lab
runs:

```
what 4_Validate_stage_tables.sql reports:
BATCH_ID
sales_2013_01_01.csv    100000
```

One batch, 100,000 rows. Correct, and it confirms something real: every line
arrived, once, from the file we meant to load. Re-run the COPY and this becomes
200,000 — so it does catch double-loads, which is worth having.

Here is the same table grouped by the dates it actually contains:

```
what the rows actually are:
TRANS_DT
2009    25761
2010    25569
2011    23679
2012    24991
```

Four years. The `BATCH_ID` says `2013_01_01`; not one row is from 2013.

Both queries ran against the same successfully loaded table. The difference is
that one groups by a label that came from the filename and the other groups by a
value that came from the data. `4_Validate_stage_tables.sql` answers *did the
load complete*. It cannot answer *is this the data I expected*, because it never
looks at the data.

Which is the summary of this worksheet. The four FILE_FORMAT settings each
handle one specific fact about this file — where the header is, how text is
quoted, which end of the date comes first — and getting any of them wrong
produces a table that loads, validates, and lies. Row counts confirm arrival.
Everything else you have to check on purpose: distinct values on text columns, a
date range, a duplicate count on anything you will join by.